<div dir="rtl">

### 1. הכנת הנתונים והסביבה 
- [ ] התקנת הספריות הנדרשות לפיתוח
- [ ] הורדת ה-Dataset מ-Kaggle
- [ ] טיפול בערכים חסרים בדאטה
- [ ] בחירת 5–7 פיצ'רים + עמודת Target

---

### 2. אימון המודל וה-Pipeline 
- [ ] המרת משתנים קטגוריאליים לערכים נומריים
- [ ] פיצול הנתונים ל-Train ו-Test
- [ ] בניית Pipeline המשולב StandardScaler ו-SVC
- [ ] אימון ה-Pipeline על נתוני האימון (fit)
- [ ] חישוב מטריקות הערכה (Accuracy, Confusion Matrix, Classification Report)
- [ ] שמירת ה-Pipeline המאומן לקובץ

---

### 3. שרת FastAPI 
- [ ] הקמת תשתית השרת ב-FastAPI
- [ ] מנגנון טעינת קובץ המודל בעליית השרת (כולל טיפול במקרה של קובץ חסר)
- [ ] יצירת Endpoint להחזרת נתוני ומטריקות המודל (`/model/info`)
- [ ] יצירת Endpoint לקבלת נתוני משתמש והחזרת חיזוי (`/predict`)

---

### 4. ממשק המשתמש (HTML / Frontend) 
- [ ] בניית דף נתוני המודל והצגת המטריקות דרך ה-API
- [ ] בניית דף טופס קליטת נתונים מהמשתמש
- [ ] חיבור כפתור ה-Check ל-API להרצת החיזוי
- [ ] הצגת תוצאת החיזוי הוויזואלית (מאושר / לא מאושר)

---

### 5. איכות, בדיקות והגשה 
- [ ] בדיקת התנהגות המערכת כשקובץ המודל חסר
- [ ] כתיבת תיעוד (Docstrings) בקוד
- [ ] הוספת לוגים (רשות - בונוס)
- [ ] העלאת האפליקציה ל-Render (רשות - בונוס)
- [ ] העלאת הקוד והמודל ל-GitHub

</div>

In [19]:
# ספריות
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.model_selection import GridSearchCV



In [7]:
# loanding data 
db = pd.read_csv('loan_data_v2.csv')
# no null values
display(db.isna().sum())


person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

['OTHER' 'RENT' 'OWN' 'MORTGAGE']


{'RENT': 1, 'OWN': 2, 'MORTGAGE': 3, 'OTHER': 4}

In [18]:
print(f'The number of approve lon: {sum(db['loan_status']==1)}')
print(f'The number of disapprove lon: {sum(db['loan_status']==0)}')

The number of approve lon: 2671
The number of disapprove lon: 9329


In [9]:
# drop weights and copy the data
db2 = db.copy()
db2 = db2.drop(['person_age','person_gender','cb_person_cred_hist_length','person_education','loan_intent','person_home_ownership'], axis = 1)

#normalisition
db2 = pd.get_dummies(db2, columns=['previous_loan_defaults_on_file'], drop_first=True)
db2

,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,credit_score,loan_status,previous_loan_defaults_on_file_Yes
0,29958.61,13,4923.19,15.32,0.16,608,0,True
1,103953.94,4,35000.00,15.38,0.34,608,1,False
2,51507.44,13,17642.82,16.54,0.34,501,0,True
3,76598.94,13,12694.67,14.87,0.17,633,0,False
4,44134.42,3,12462.46,15.61,0.28,588,0,False
...,...,...,...,...,...,...,...,...
11995,71097.07,23,11086.21,17.18,0.16,620,0,True
11996,26371.24,1,13540.44,16.78,0.51,592,0,True
11997,27586.56,20,7764.84,19.17,0.28,585,0,False
11998,64217.59,10,13853.73,17.88,0.22,553,0,True


In [22]:
#model svm
X = db2.drop('loan_status', axis=1).values
y = db2['loan_status'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

C_range = [0.1, 1, 10]
best_score = 0
best_C = None
best_pipeline = None

for c_val in C_range:
    loan_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=c_val))
    ])

    scores = cross_val_score(
        loan_pipeline, 
        X_train, 
        y_train, 
        cv=5, 
        scoring='accuracy'
    )
    mean_score = np.mean(scores)
    print(f"Testing C={c_val} | Mean Accuracy: {mean_score:.4f}")
    
    if mean_score > best_score:
        best_score = mean_score
        best_C = c_val
        best_pipeline = loan_pipeline
print(f"The winning parameter is C={best_C} with an accuracy of {best_score:.4f}")
best_pipeline.fit(X_train, y_train)
y_pred_best = best_pipeline.predict(X_test)
print("\nAccuracy on Test Set:", accuracy_score(y_test, y_pred_best))

cm = confusion_matrix(y_test, y_pred_best)
print("Confusion matrix:\n", cm)
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Classification Report:\n")
print(classification_report(y_test, y_pred_best))

Testing C=0.1 | Mean Accuracy: 0.8545
Testing C=1 | Mean Accuracy: 0.8528
Testing C=10 | Mean Accuracy: 0.8521
The winning parameter is C=0.1 with an accuracy of 0.8545

Accuracy on Test Set: 0.8633333333333333
Confusion matrix:
 [[1726  140]
 [ 188  346]]
Accuracy: 0.8633333333333333
Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.92      0.91      1866
           1       0.71      0.65      0.68       534

    accuracy                           0.86      2400
   macro avg       0.81      0.79      0.80      2400
weighted avg       0.86      0.86      0.86      2400



In [49]:
import joblib

# שמירת ה-Pipeline המאומן לקובץ
joblib.dump(best_pipeline, 'loan_model.pkl')

print("המודל נשמר בהצלחה לקובץ!")

המודל נשמר בהצלחה לקובץ!


In [ ]:
param_grid = {
    'model__C': [0.1, 1, 10, 100],
    'model__gamma': [0.01, 0.1, 1, 10],
    'model__kernel': ['linear', 'rbf', 'poly', 'sigmoid']
}

grid = GridSearchCV(
    loan_pipeline,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1  
)

grid.fit(X, y)

print("Best parameters:")
print(grid.best_params_)

print("\nBest cross-validation accuracy:")
print(grid.best_score_)